In [1]:
import pandas as pd
import numpy as np

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    KBinsDiscretizer,
    OneHotEncoder,
    PowerTransformer,
    StandardScaler,
)
from sklearn.impute import SimpleImputer

from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    ConfusionMatrixDisplay,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
    roc_auc_score,
)

import warnings; warnings.filterwarnings("ignore")

## Read Data CSV

In [2]:
sessions = pd.read_csv("Raw_Data/challenge2_train.csv")

display(sessions.head())
sessions.info()

print()
display(sessions["Revenue"].value_counts(dropna=False))
print()
display(sessions["Revenue"].value_counts(normalize=True, dropna=False))

,ID,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,1,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,3,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
2,4,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
3,7,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.4,Feb,2,4,3,3,Returning_Visitor,False,False
4,9,0,0.0,0,0.0,2,37.000000,0.00,0.10,0.0,0.8,Feb,2,2,2,3,Returning_Visitor,False,False


<class 'pandas.DataFrame'>
RangeIndex: 9247 entries, 0 to 9246
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID                       9247 non-null   int64  
 1   Administrative           9247 non-null   int64  
 2   Administrative_Duration  9247 non-null   float64
 3   Informational            9247 non-null   int64  
 4   Informational_Duration   9247 non-null   float64
 5   ProductRelated           9247 non-null   int64  
 6   ProductRelated_Duration  9247 non-null   float64
 7   BounceRates              9247 non-null   float64
 8   ExitRates                9247 non-null   float64
 9   PageValues               9247 non-null   float64
 10  SpecialDay               9247 non-null   float64
 11  Month                    9247 non-null   str    
 12  OperatingSystems         9247 non-null   int64  
 13  Browser                  9247 non-null   int64  
 14  Region                   9247 non-n

Revenue
False    7816
True     1431
Name: count, dtype: int64

Revenue
False    0.845247
True     0.154753
Name: proportion, dtype: float64

## Train/Validation Splits 

In [3]:
X = sessions.drop(columns=["Revenue", "ID"])
y = sessions["Revenue"].copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

display(pd.DataFrame({
    "development": y_train.value_counts(normalize=True),
    "holdout": y_valid.value_counts(normalize=True),
}))

,development,holdout
Revenue,,
False,0.845208,0.845405
True,0.154792,0.154595


## Model Pipeline

In [4]:
categorical = [
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType",
    "Weekend",
]

numerical = [column for column in X_train.columns if column not in categorical]

# 2. Build leakage-free preprocessing per column type
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    # Yeo-Johnson maps skewed continuous features to Gaussian bell curves
    ('power', PowerTransformer(method='yeo-johnson', standardize=True))
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    # Group rare categories (<1% frequency) to prevent zero-variance overfitting
    ('ohe', OneHotEncoder(
        handle_unknown='infrequent_if_exist',
        min_frequency=0.01,
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numerical),
        ('cat', cat_pipeline, categorical)
    ],
    remainder='drop'
)

### Logistic Regression

In [ ]:
# --- 1. Cross-Validation & Scoring Setup ---
# Maintain class proportions (~84.5% False / ~15.5% True) across 5 folds
# and optimize specifically for the positive class (Revenue=True) F1 score
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_purchase = make_scorer(f1_score, pos_label=True, zero_division=0)

# --- 2. Dedicated Logistic Regression Base Pipeline ---
# Clone the global preprocessor so fitting GridSearchCV never mutates shared pipelines
logistic_pipeline = Pipeline([
    ("prep", clone(preprocessor)),
    ("model", LogisticRegression(max_iter=2000, random_state=42)),
])

# --- 3. Multi-Regime Regularization & Class-Weight Grid ---
# Dictionary 1 tests L1 (feature-selecting) and L2 (ridge) penalties across liblinear and saga.
# Dictionary 2 tests ElasticNet (combined L1 + L2) with saga to handle collinear numerical pairs.
# Both grids sweep custom class_weight ratios to balance precision and recall on imbalanced data.
param_grid = [
    {
        "model__solver": ["liblinear", "saga"],
        "model__penalty": ["l1", "l2"],
        "model__C": [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0],
        "model__class_weight": [
            None,
            "balanced",
            {False: 1, True: 2},
            {False: 1, True: 3},
            {False: 1, True: 4},
        ],
    },
    {
        "model__solver": ["saga"],
        "model__penalty": ["elasticnet"],
        "model__l1_ratio": [0.25, 0.5, 0.75],
        "model__C": [0.01, 0.1, 0.5, 1.0, 5.0],
        "model__class_weight": [
            None,
            "balanced",
            {False: 1, True: 2},
            {False: 1, True: 3},
            {False: 1, True: 4},
        ],
    },
]

# --- 4. Configure and Fit GridSearchCV Exclusively on Training Data ---
# Fit strictly on X_train to prevent validation data leakage; refit=True automatically
# retrains the winning hyperparameter combination on the full training set
logistic_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=param_grid,
    scoring=f1_purchase,
    cv=cv,
    refit=True,
    return_train_score=True,
    n_jobs=-1,
)
logistic_search.fit(X_train, y_train)

# --- 5. Cross-Validation Performance Summary ---
# Compare mean CV training vs. validation F1 to check generalization before holdout evaluation
best_index = logistic_search.best_index_
best_cv_train_f1 = logistic_search.cv_results_["mean_train_score"][best_index]
best_cv_valid_f1 = logistic_search.best_score_

print("Best Logistic Regression Parameters:", logistic_search.best_params_)
print(f"Best Mean CV Training F1:           {best_cv_train_f1:.3f}")
print(f"Best Mean CV Validation F1:         {best_cv_valid_f1:.3f}")

# --- 6. Learned Feature Weight Inspection ---
# Extract the refitted winning pipeline and display the top coefficients sorted by magnitude
best_logistic_pipe = logistic_search.best_estimator_
feature_names = best_logistic_pipe.named_steps["prep"].get_feature_names_out()
weights = best_logistic_pipe.named_steps["model"].coef_[0]

logistic_weights = (
    pd.DataFrame({
        "Feature": feature_names,
        "Weight": weights,
    })
    .assign(Abs_Weight=lambda df: df["Weight"].abs())
    .sort_values(by="Abs_Weight", ascending=False)
    .drop(columns=["Abs_Weight"])
    .reset_index(drop=True)
)

display(logistic_weights.head(15).round(3))

In [ ]:
# --- 1. Generate Predictions on Training and Holdout Sets ---
# Use the refitted best Logistic Regression pipeline for class and probability predictions
y_train_pred = best_logistic_pipe.predict(X_train)
y_valid_pred = best_logistic_pipe.predict(X_valid)
y_valid_prob = best_logistic_pipe.predict_proba(X_valid)[:, 1]

# --- 2. Build Classification Metrics Comparison Table ---
# Evaluate Accuracy, Precision, Recall, and F1 Score side-by-side on train vs. validation
metrics_comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Training": [
        accuracy_score(y_train, y_train_pred),
        precision_score(y_train, y_train_pred, zero_division=0),
        recall_score(y_train, y_train_pred, zero_division=0),
        f1_score(y_train, y_train_pred, zero_division=0),
    ],
    "Validation": [
        accuracy_score(y_valid, y_valid_pred),
        precision_score(y_valid, y_valid_pred, zero_division=0),
        recall_score(y_valid, y_valid_pred, zero_division=0),
        f1_score(y_valid, y_valid_pred, zero_division=0),
    ],
})

display(metrics_comparison.round(3))

### Naïve Bayes

In [ ]:
# --- 1. Cross-Validation & Scoring Setup ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_purchase = make_scorer(f1_score, pos_label=True, zero_division=0)

# --- 2. Dedicated Naïve Bayes Pipeline & Discretized Numerical Sub-Pipeline ---
# Clone the global preprocessor so overriding 'prep__num' never mutates shared pipelines
Bayes_pipe = Pipeline([
    ("prep", clone(preprocessor)),
    ("clf", GaussianNB()),
])

discretized_num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("bins", KBinsDiscretizer(encode="onehot-dense", subsample=None)),
])

# --- 3. Multi-Regime Hyperparameter Grid ---
# Regime A: Discretized numerical bins + BernoulliNB (binarize=None)
# Regime B: Default PowerTransformer + BernoulliNB across non-negative z-score cutoffs
# Regime C: Default PowerTransformer + continuous GaussianNB
param_grid = [
    {
        "prep__num": [discretized_num_pipeline],
        "prep__num__bins__n_bins": [3, 5, 8, 10],
        "prep__num__bins__strategy": ["quantile", "uniform"],
        "clf": [BernoulliNB(binarize=None)],
        "clf__alpha": [0.01, 0.1, 0.5, 1.0, 2.0],
        "clf__class_prior": [None, [0.8, 0.2], [0.7, 0.3], [0.6, 0.4], [0.5, 0.5]],
    },
    {
        "clf": [BernoulliNB()],
        "clf__alpha": [0.01, 0.1, 0.5, 1.0, 2.0],
        "clf__binarize": [0.0, 0.25, 0.5, 1.0],
        "clf__class_prior": [None, [0.8, 0.2], [0.7, 0.3], [0.6, 0.4], [0.5, 0.5]],
    },
    {
        "clf": [GaussianNB()],
        "clf__var_smoothing": np.logspace(-11, -1, 11),
        "clf__priors": [None, [0.8, 0.2], [0.7, 0.3], [0.6, 0.4], [0.5, 0.5]],
    },
]

# --- 4. Fit GridSearchCV Exclusively on Training Data ---
bayes_search = GridSearchCV(
    estimator=Bayes_pipe,
    param_grid=param_grid,
    scoring=f1_purchase,
    cv=cv,
    refit=True,
    return_train_score=True,
    n_jobs=-1,
)
bayes_search.fit(X_train, y_train)

# --- 5. Cross-Validation Performance Summary ---
best_index = bayes_search.best_index_
best_cv_train_f1 = bayes_search.cv_results_["mean_train_score"][best_index]
best_cv_valid_f1 = bayes_search.best_score_

print("Best GridSearch Parameters:", bayes_search.best_params_)
print(f"Best Mean CV Training F1:   {best_cv_train_f1:.3f}")
print(f"Best Mean CV Validation F1: {best_cv_valid_f1:.3f}")

# --- 6. Dynamic Learned Parameter Inspection ---
best_bayes_pipe = bayes_search.best_estimator_
feature_names = best_bayes_pipe.named_steps["prep"].get_feature_names_out()
best_clf = best_bayes_pipe.named_steps["clf"]

if isinstance(best_clf, GaussianNB):
    learned_params = pd.DataFrame({
        "Feature": feature_names,
        "Mean (y=0)": best_clf.theta_[0],
        "Mean (y=1)": best_clf.theta_[1],
        "Var (y=0)": best_clf.var_[0],
        "Var (y=1)": best_clf.var_[1],
    })
else:
    feature_prob = np.exp(best_clf.feature_log_prob_)
    learned_params = pd.DataFrame({
        "Feature": feature_names,
        "P(Feature=1 | y=0)": feature_prob[0],
        "P(Feature=1 | y=1)": feature_prob[1],
    })

display(learned_params.head(10).round(3))

In [ ]:
# Predictions for training and validation sets using the tuned best pipeline
y_train_pred = best_bayes_pipe.predict(X_train)
y_valid_pred = best_bayes_pipe.predict(X_valid)
y_valid_prob = best_bayes_pipe.predict_proba(X_valid)[:, 1]

# Compare the main classification metrics
metrics_comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Training": [
        accuracy_score(y_train, y_train_pred),
        precision_score(y_train, y_train_pred, zero_division=0),
        recall_score(y_train, y_train_pred, zero_division=0),
        f1_score(y_train, y_train_pred, zero_division=0),
    ],
    "Validation": [
        accuracy_score(y_valid, y_valid_pred),
        precision_score(y_valid, y_valid_pred, zero_division=0),
        recall_score(y_valid, y_valid_pred, zero_division=0),
        f1_score(y_valid, y_valid_pred, zero_division=0),
    ],
})

display(metrics_comparison.round(3))

### K-Nearest Neighbors

In [ ]:
knn_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", KNeighborsClassifier(n_neighbors=5)),
])

fitted_knn = clone(knn_pipeline).fit(X_train, y_train)
knn_model = fitted_knn.named_steps["model"]
print(knn_model.weights)

session = X_train.iloc[[0]]
transformed_session = fitted_knn.named_steps["prep"].transform(session)
distances, indices = knn_model.kneighbors(transformed_session)

display(pd.DataFrame({
    "Distance": distances[0],
    "Purchased": y_train.iloc[indices[0]].to_numpy(),
}))

## Support Vector Machine

In [ ]:

svmlinear_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", SVC(kernel="linear", C=1.0, class_weight="balanced"))
])

svmrbf_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        class_weight="balanced"
    ))
])

fitted_svmlinear = clone(svmlinear_pipe).fit(X_train, y_train)
fitted_svmrbf = clone(svmrbf_pipe).fit(X_train, y_train)

svmlinear_model = fitted_svmlinear.named_steps["model"]
svmrbf_model = fitted_svmrbf.named_steps["model"]

print("Linear SVM support vectors:", svmlinear_model.n_support_)
print("RBF SVM support vectors:", svmrbf_model.n_support_)


## Decision Tree

In [ ]:
dt_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", DecisionTreeClassifier(
        random_state=42,
        max_depth=5,
        min_samples_leaf=11,
    ))
])
fitted_dt = clone(dt_pipe).fit(X_train, y_train)

dt_model = fitted_dt.named_steps["model"]
dt_importance = pd.DataFrame({
    "Feature": fitted_dt.named_steps["prep"].get_feature_names_out(),
    "Importance": dt_model.feature_importances_
})

display(
    dt_importance.sort_values(
        "Importance",
        ascending=False
    ).head(10).round(3)
)


## Random Forest

In [ ]:
rf_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=400,
        random_state=42,
    ))
])

fitted_rf = clone(rf_pipe).fit(X_train, y_train)
rf_model = fitted_rf.named_steps["model"]

feature_names = fitted_rf.named_steps["prep"].get_feature_names_out()

importances = fitted_rf.named_steps["model"].feature_importances_

rf_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

display(
    rf_importance.sort_values(
        "Importance",
        ascending=False
    ).head(10).round(3)
)


## Gradient Boost

In [ ]:
gb_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", GradientBoostingClassifier(
        n_estimators=170,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ))
])

fitted_gb = clone(gb_pipe).fit(X_train, y_train)
gb_model = fitted_gb.named_steps["model"]
gb_importance = pd.DataFrame({
    "Feature": fitted_gb.named_steps["prep"].get_feature_names_out(),
    "Importance": gb_model.feature_importances_
})

display(
    gb_importance.sort_values(
        "Importance",
        ascending=False
    ).head(10).round(3)
)


## Model Comparison

In [ ]:
model_results = []

pipelines = {
    "Logistic Regression": logistic_pipeline,
    "Naïve Bayes": best_bayes_pipe,
    "K-Nearest Neighbors": knn_pipeline,
    "Linear SVM": svmlinear_pipe,
    "RBF SVM": svmrbf_pipe,
    "Decision Tree": dt_pipe,
    "Random Forest": rf_pipe,
    "Gradient Boost": gb_pipe
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_purchase = make_scorer(f1_score, pos_label=True, zero_division=0)

for model, pipeline in pipelines.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=f1_purchase,
        return_train_score=True,
    )

    model_results.append({
        "Model": model,
        "Mean training F1": scores["train_score"].mean(),
        "Mean CV F1": scores["test_score"].mean(),
    })

display(pd.DataFrame(model_results).round(3))